# RSNA Knee ConvNeXtV2 domain-pool screen
Private leakage-safe validation of an attributed public RSNA-trained backbone. No submission.


In [ ]:
from __future__ import annotations
import os
os.environ['RSNA_IMG'] = '288'
os.environ['RSNA_CROP_MM'] = '140'
os.environ['RSNA_N_GROUP_MAX'] = '4'
os.environ['RSNA_WINDOW'] = '0.08,0.92'
"""RSNA knee: study-level 12-label pipeline.

Synthesised from two community notebooks, with the parts that were wrong or missing in
both replaced. What is inherited, and from where:

  from `rsna-knee-baseline-v1`   header-recovered slot scheme, constant-physical-scale
                                 sampling, laterality normalisation, uint8 cache,
                                 per-diagnosis slot attention, report-hash grouping
  from `rsna-knee-eda-to-2-5d`   protocol-only features as a legal test-time signal

What is new here is listed in `README.md`; the five that change results are grouped
K-fold instead of a single holdout, gold studies held out of their own fold so the
annotated reference is honest, anatomy-preserving augmentation, a backbone that refuses
to train from random initialisation silently, and rank fusion of the folds and the
protocol model.
"""


import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F


T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

class CFG:
    seed = 2026               # folds, target construction, text distillation

    # Separate from `seed` on purpose. This one drives only the stochastic parts of
    # image-model training: head init, batch order, group choice, augmentation. Bump
    # it alone and the folds and the derived targets are byte-identical, so the change
    # in the OOF score is training noise and nothing else. That number is the gate an
    # experiment has to clear before its result means anything.
    train_seed = int(os.environ.get("RSNA_TRAIN_SEED", "2026"))

    # Encoder input, a multiple of the DINOv2 patch size 14. Overridable because `img`
    # and `crop_mm` are two knobs on ONE ratio - what a single token covers in
    # millimetres, 14*crop_mm/img. exp-026 moved that ratio 10.00 -> 8.13 mm/token by
    # shrinking the crop, for free. Raising img moves the same ratio but costs compute
    # quadratically, so the two must be tested separately or the result is confounded:
    #   160mm @ 224 = 10.00 mm/token   (pre-exp-026)
    #   130mm @ 224 =  8.13            (champion, 1.0x compute)
    #   130mm @ 336 =  5.42            (2.25x compute)
    img = int(os.environ.get("RSNA_IMG", "224"))

    # Physical extent of the centre crop. 160 is the historical default and is also
    # exactly the measured MEDIAN field of view of this corpus, which is the worst
    # possible place to stand: the guard below reduces to `FOV > crop_mm`, so a
    # threshold sitting at the median disables the crop on about half the corpus by
    # construction. Measured: 61.1% of slots skipped.
    # Lowering it is the OTHER fix for that defect. Padding (pad_short_fov) grows the
    # image to meet the crop and priced at -0.0012, inside the floor; shrinking the
    # crop makes it apply everywhere instead, and buys a finer pixel pitch for free
    # since pitch = crop_mm / img. Overridable so both fixes share one control.
    crop_mm = float(os.environ.get("RSNA_CROP_MM", "160"))

    # The crop guard `want < min(h, w)` reduces exactly to `FOV > crop_mm`, so a series
    # whose field of view is at or below crop_mm was silently left unnormalised - and
    # the measured median FOV in this corpus is 160 mm, right at the threshold. Padding
    # to crop_mm fixes the scale for those. Off by default so it can be A/B'd against
    # the noise floor rather than smuggled in.
    pad_short_fov = os.environ.get("RSNA_PAD_SHORT_FOV", "0") == "1"

    # Average the last N epochs instead of selecting the best-scoring one.
    # 0 restores the old select-on-the-scored-fold behaviour, so the two are
    # A/B-able. See the note in train_fold for why this exists.
    swa_epochs = int(os.environ.get("RSNA_SWA_EPOCHS", "0"))

    # Whether to accept the geometric laterality fallback (sign of ImagePositionPatient
    # x). Off by default: the tag, ImageLaterality and text sources are all explicit
    # statements by the scanner, while this one is an inference. The run logs how often
    # it agrees with the tag where both exist, so it can be switched on with evidence
    # rather than on faith.
    lat_from_geometry = os.environ.get("RSNA_LAT_GEOMETRY", "0") == "1"
    group = 3                 # slices per encoder input, stacked as the three channels
    # Groups per slot, before the memory budget is applied. Exposed so slice
    # coverage can be priced DOWNWARD: if 1 group (3 slices) scores like 3 groups
    # (9 slices), the curve is already flat and "look at more slices" is dead
    # before it is built.
    n_group_max = int(os.environ.get("RSNA_N_GROUP_MAX", "3"))
    cache_budget_gb = 12.0
    ram_fraction = 0.45       # ceiling on the cache as a share of free RAM

    hdr_threads = 16
    pix_threads = 12

    n_folds = 5
    max_folds_to_run = int(os.environ.get("RSNA_FOLDS", "5"))

    # 12 was inherited from the first version of this pipeline and has never been varied
    # in ~15 experiments, all of which changed how the PICTURES are prepared. The public
    # 0.891 notebook trains 30 (entry 037). Undertraining would be invisible to every
    # experiment run so far, because it affects control and arm identically.
    epochs = int(os.environ.get("RSNA_EPOCHS", "12"))
    # 0 = OneCycle (the historical schedule). >0 = cosine warm restarts of this length.
    cycle_epochs = int(os.environ.get("RSNA_CYCLE_EPOCHS", "0"))
    batch_studies = 8
    lr_backbone = 8e-6        # the encoder is being adapted, not retrained
    lr_head = 1e-3
    weight_decay = 0.02
    unfreeze_last = 6         # trainable transformer blocks, from the output end
    eval_batch = 12
    time_budget = float(os.environ.get("RSNA_TIME_BUDGET", 8.0 * 3600))

    w_text = 0.35             # weight of the distilled text model in the label blend
    w_protocol = 0.10         # weight of the protocol-only model in the final fusion
    gold_weight = 3.0


# Six slots: three planes crossed with the acquisition axes. The fat-suppressed
# fluid-sensitive series exist for nearly every study; the T1 and the non-suppressed
# fluid-sensitive series are scarcer, which is what the presence mask is for.
SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

# The alternative: plane x the provided flag, ignoring the recovered weighting.
#
# Worth stating why the recovered scheme is the default. The EDA notebook found that
# `Fluid_Sensitive` and `Fat_Suppression` are byte-identical across all 24,371 series
# rows. They name two physically independent properties - contrast weighting, set by
# TR/TE, and a fat-suppression preparation applied on top of any weighting - so two
# identical columns carry one bit between them, not two. Recovering both axes from the
# DICOM header is therefore not a refinement; it is the difference between six slots
# and three.
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


def seed_all(seed=CFG.seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def find_root(explicit=None):
    if explicit is not None:
        p = Path(explicit)
        if (p / "test.csv").is_file():
            return p
        raise FileNotFoundError(f"no test.csv under {p}")
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        # last resort: two-level scan, because the mount is sometimes nested one deeper
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError("competition mount not found")


def find_dinov2(variant="small"):
    """Locate a mounted DINOv2 checkpoint directory by variant name."""
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None

HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Laterality fallbacks. `Laterality` is absent on about half the series,
            # and a study with no side at all is never mirrored - so medial and lateral
            # sit on opposite sides of the image for left and right knees. Four of the
            # twelve targets are side-specific, so that is a third of the metric.
            "ImageLaterality", "StudyDescription", "BodyPartExamined",
            "PatientPosition", "ImagePositionPatient"]


def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(root, split):
    base = Path(root) / split
    items = []
    if not base.is_dir():
        return pd.DataFrame()
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=CFG.hdr_threads) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header.

    Two string-matching cautions, both of which silently invert the answer if missed:
    underscore is a word character, so a token test for `we` (water excitation) never
    fires inside `t2_de3d_we_tra` unless separators are normalised first; and GE writes
    `SAT_GEMS` for *spatial* saturation, so ScanOptions must be matched as exact tokens
    or non-fat-suppressed series get marked as suppressed.
    """
    if df.empty:
        return df
    for t in HDR_TAGS:
        if t not in df.columns:
            df[t] = None

    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1 = desc.str.contains(_T1_RX)
    t2 = desc.str.contains(_T2_RX)
    pdw = desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler benefits from the margin.
    """
    if series_df.empty:
        return {}
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel = sel & (g["fluid"] == fluid)
            cand = g[sel]
            if len(cand) == 0 and fluid is False:
                # T1 slots are the scarcest; fall back to any non-fat-sat series in the
                # plane before giving up on the slot entirely.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out

ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "SliceLocation",
              "InstanceNumber"]

# Ordering happens on reader threads, so `d[k] += 1` - three bytecodes, not one - needs
# the lock. An undercounted fallback reads as a clean run.
import threading

ORDER_STATS = {"geometry": 0, "slice_location": 0, "instance_number": 0, "filename": 0}
_ORDER_LOCK = threading.Lock()

# The pixel path had no instrumentation at all, so three separate silent failures were
# invisible: a physical crop that skips itself, a slice that fails to decode and is
# substituted with black, and an inverted photometric interpretation. Each one degrades
# an image without raising anything. Counted here, they become measurements.
PIX_STATS = {"crop_applied": 0, "crop_padded": 0, "crop_skipped_short_fov": 0,
             "crop_no_spacing": 0, "decode_failed": 0, "mono1_inverted": 0,
             "slices_read": 0}
_PIX_LOCK = threading.Lock()


def _bump(key):
    with _ORDER_LOCK:
        ORDER_STATS[key] += 1


def _bump_pix(key, n=1):
    with _PIX_LOCK:
        PIX_STATS[key] += n


def order_slices(directory, files):
    """Return `files` sorted by position along the stack, nearest-first.

    The filenames in this corpus are SOP Instance UIDs, which are random, so the sorted
    filename order is a random permutation of the anatomy. Measured on the competition
    data, filename order agrees with physical order on about 5% of slices.

    That is not a cosmetic problem. Two things downstream assume the list is ordered:
    the sampling window ("the central portion of the stack") and the grouping of
    adjacent slices into the encoder's three channels. Under a random permutation the
    window selects a uniform random sample of the whole series rather than its centre,
    and a group of three adjacent entries is three unrelated positions in the knee - so
    the 2.5D input carried no depth information at all.

    Fallback chain, because not every series records the same tags:
      1. project ImagePositionPatient onto the normal of ImageOrientationPatient. This
         is the only one that is correct for an obliquely angled stack.
      2. SliceLocation, which the scanner has already projected but which is signed
         inconsistently between vendors.
      3. InstanceNumber, acquisition order - right for most stacks, wrong for
         interleaved acquisitions.
      4. filename, i.e. give up, and say so in the counters.
    """
    if len(files) < 2:
        return list(files)

    positions, locations, instances = [], [], []
    normal = None
    for name in files:
        try:
            ds = pydicom.dcmread(os.path.join(directory, name), stop_before_pixels=True,
                                 force=True, specific_tags=ORDER_TAGS)
        except Exception:
            positions.append(None)
            locations.append(None)
            instances.append(None)
            continue
        pos = getattr(ds, "ImagePositionPatient", None)
        orient = getattr(ds, "ImageOrientationPatient", None)
        if normal is None and orient is not None and len(orient) == 6:
            try:
                row_dir = np.array([float(v) for v in orient[:3]])
                col_dir = np.array([float(v) for v in orient[3:]])
                normal = np.cross(row_dir, col_dir)
            except Exception:
                normal = None
        try:
            positions.append(np.array([float(v) for v in pos]) if pos is not None else None)
        except Exception:
            positions.append(None)
        loc = getattr(ds, "SliceLocation", None)
        locations.append(float(loc) if loc is not None else None)
        num = getattr(ds, "InstanceNumber", None)
        instances.append(float(num) if num is not None else None)

    def sorted_by(values, stat):
        _bump(stat)
        return [f for _, f in sorted(zip(values, files), key=lambda pair: pair[0])]

    if normal is not None and all(p is not None for p in positions):
        return sorted_by([float(p @ normal) for p in positions], "geometry")
    if all(loc is not None for loc in locations):
        return sorted_by(locations, "slice_location")
    if all(num is not None for num in instances):
        return sorted_by(instances, "instance_number")
    _bump("filename")
    return list(files)


# Fraction of the ordered stack each plane is sampled across.
#
# These are wider than the 20-80% window this replaced, and the widening is part of the
# ordering fix rather than a separate change. Before ordering, "the central 60%" of a
# random permutation was a uniform sample of the *whole* series; applying that same
# window to a correctly ordered stack would genuinely discard the outer 40% and quietly
# bundle a coverage reduction into this change. These keep the physical span roughly as
# it was, so the ordering is the only variable that moved.
#
# The per-plane split follows Will's (`wguesdon`) reasoning: the menisci sit at the
# medial and lateral extremes of a sagittal stack and Baker's cysts sit posteriorly on
# an axial one, so those planes need their edges; a coronal stack's useful
# anterior-posterior range is narrower.
PLANE_WINDOW = {"Sagittal": (0.10, 0.90), "Axial": (0.10, 0.90),
                "Coronal": (0.15, 0.85)}


def read_slot(rec, n_slice, out_size, plane=None, group=None):
    """`n_slice` slices from one series, physically ordered, at `out_size` pixels.

    Returns uint8 [n_slice, out, out], normalised per-series to its 1st-99th percentile.
    Percentiles rather than min/max because MR intensity has no absolute scale and one
    bright vessel would otherwise compress the whole dynamic range.

    The slices come out as `n_slice // group` anchors spread across the sampling window,
    each anchor contributing `group` *physically adjacent* slices. That layout is what
    `take_group` slices back out into the encoder's three channels, so a channel triplet
    is a genuine depth neighbourhood - roughly 10 mm of knee at this corpus's slice gaps
    - rather than three arbitrary positions. Spreading all nine slices evenly instead
    would cover more of the joint but hand the encoder three views 20 mm apart, which is
    a slab, not a 2.5D triplet.

    A slice of N pixels at spacing s mm covers Ns millimetres. Both vary widely here, so
    a fixed-pixel resize hands the encoder images whose physical scale differs by about
    3x. Cropping to a constant physical extent first fixes the effective scale at
    crop_mm / img mm per pixel regardless of acquisition.
    """
    files, d, px = rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    group = CFG.group if group is None else group

    files = order_slices(d, files)

    # exp-016 measured 3 central slices BEATING 9 spread slices by +0.0086 (4x the
    # 0.0020 floor). With one anchor the code takes the window CENTRE; with three it
    # spreads them to the window edges - so "fewer slices" and "more central slices" are
    # confounded in that result. RSNA_WINDOW narrows the window without changing the
    # count, which separates them.
    lo_f, hi_f = PLANE_WINDOW.get(plane, (0.10, 0.90))
    if os.environ.get("RSNA_WINDOW"):
        lo_f, hi_f = (float(x) for x in os.environ["RSNA_WINDOW"].split(","))
    lo, hi = int(lo_f * (n - 1)), int(hi_f * (n - 1))
    if hi <= lo:
        lo, hi = 0, n - 1

    n_anchor = max(1, n_slice // group)
    anchors = (np.linspace(lo, hi, n_anchor).astype(int) if n_anchor > 1
               else np.array([(lo + hi) // 2]))

    idx = []
    for centre in anchors:
        # Adjacent slices around the anchor, clipped into the series rather than
        # wrapped: a wrap would put the far end of the knee in the same channel stack.
        start = int(np.clip(centre - group // 2, 0, max(0, n - group)))
        idx.extend(range(start, min(start + group, n)))
    while len(idx) < n_slice:
        idx.append(idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
            # MONOCHROME1 means high value renders as black - the image is inverted
            # relative to MONOCHROME2. Left uncorrected, fluid reads dark in those
            # series while reading bright everywhere else, and Effusion and Synovitis
            # are fluid-bright findings. Invert into the MONOCHROME2 convention.
            if str(getattr(ds, "PhotometricInterpretation", "")).strip() == "MONOCHROME1":
                a = a.max() - a
                _bump_pix("mono1_inverted")
            _bump_pix("slices_read")
        except Exception:
            a = None
            _bump_pix("decode_failed")
        planes.append(a)

    shp = next((p.shape for p in planes if p is not None), None)
    if shp is None:
        return None
    planes = [p if (p is not None and p.shape == shp) else np.zeros(shp, np.float32)
              for p in planes]
    vol = np.stack(planes)

    if px and np.isfinite(px) and px > 0:
        want = int(round(CFG.crop_mm / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]
            _bump_pix("crop_applied")
        elif want >= min(h, w):
            # The series covers less than crop_mm, so there is nothing to crop away.
            # Resizing here would set mm/px from the acquisition instead of from
            # crop_mm, which is the one thing the constant crop exists to prevent.
            # Padding to the same physical extent keeps the scale fixed; the border is
            # black because no anatomy was acquired there.
            if CFG.pad_short_fov:
                py, pxd = max(0, want - h), max(0, want - w)
                vol = np.pad(vol, ((0, 0), (py // 2, py - py // 2),
                                   (pxd // 2, pxd - pxd // 2)))
                _bump_pix("crop_padded")
            else:
                _bump_pix("crop_skipped_short_fov")
        else:
            _bump_pix("crop_skipped_short_fov")
    else:
        _bump_pix("crop_no_spacing")

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32: intensity is already normalised into [0, 1] here, so eight bits
    # cost nothing a bilinear resize has not already cost, and the cache is a quarter the
    # size for it.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


# Word-boundary side markers. Deliberately NOT including bare Turkish "sag" (right):
# every sagittal protocol in this corpus is named `sag_...`, so that token would mark
# most of the dataset as right knees. Same family of trap as GE's SAT_GEMS. The
# diacritic form is unambiguous and is matched; the stripped form is not.
_SIDE_RX = [
    ("R", re.compile(r"\b(right|rt|r_?knee|knee_?r|dexter|sağ|derech[ao]|rechts?|"
                     r"droite?|δεξ\w*)\b", re.I)),
    ("L", re.compile(r"\b(left|lt|l_?knee|knee_?l|sinister|sol|izquierd[ao]|links?|"
                     r"gauche|αριστερ\w*)\b", re.I)),
]

LAT_STATS = {"tag": 0, "image_laterality": 0, "text": 0, "geometry": 0, "unknown": 0,
             "geom_agree": 0, "geom_disagree": 0}


def _lat_from_text(*fields):
    """Side from any free-text header field, or None if absent or contradictory."""
    blob = " ".join(str(f) for f in fields if f and str(f).lower() != "nan")
    blob = re.sub(r"[^\w\s]", " ", blob)
    hits = {side for side, rx in _SIDE_RX if rx.search(blob)}
    return hits.pop() if len(hits) == 1 else None


def _lat_from_geometry(ipp):
    """Side from the x coordinate of ImagePositionPatient.

    DICOM's patient coordinate system is LPS: +x runs toward the patient's LEFT. A knee
    is therefore centred at negative x when it is the right knee. This is independent of
    head-first/feet-first, because ImagePositionPatient is already expressed in patient
    coordinates rather than scanner coordinates.

    The threshold exists because a value near zero means the scan is near the midline,
    where the sign carries no information.
    """
    try:
        x = float(str(ipp).split("|")[0])
    except (TypeError, ValueError):
        return None
    if abs(x) < 20.0:
        return None
    return "R" if x < 0 else "L"


def resolve_laterality(g):
    """Study -> 'L'/'R'/None, from the strongest available evidence.

    Order: the explicit tag, then ImageLaterality, then side words in any description
    field, then the scan geometry. Each source is counted so the log says how much of
    the corpus each one actually rescued, and the geometric rule is scored against the
    explicit tag wherever both exist - a fallback nobody has validated is a guess.
    """
    vals = [str(x).strip().upper() for x in g.get("Laterality", pd.Series(dtype=object))
            .dropna()]
    vals = [v[0] for v in vals if v and v[0] in ("L", "R")]

    geom = next((s for s in (_lat_from_geometry(v)
                             for v in g.get("ImagePositionPatient",
                                            pd.Series(dtype=object)).dropna()) if s), None)
    if vals and geom:
        _bump_lat("geom_agree" if geom == vals[0] else "geom_disagree")
    if vals:
        _bump_lat("tag")
        return vals[0]

    ivals = [str(x).strip().upper() for x in
             g.get("ImageLaterality", pd.Series(dtype=object)).dropna()]
    ivals = [v[0] for v in ivals if v and v[0] in ("L", "R")]
    if ivals:
        _bump_lat("image_laterality")
        return ivals[0]

    txt = _lat_from_text(*g.get("SeriesDescription", pd.Series(dtype=object)).tolist(),
                         *g.get("StudyDescription", pd.Series(dtype=object)).tolist(),
                         *g.get("BodyPartExamined", pd.Series(dtype=object)).tolist())
    if txt:
        _bump_lat("text")
        return txt

    if geom and CFG.lat_from_geometry:
        _bump_lat("geometry")
        return geom

    _bump_lat("unknown")
    return None


def _bump_lat(key):
    LAT_STATS[key] += 1


def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Four of the twelve targets are medial/lateral pairs, and medial is defined against
    the body midline, so which side of the *image* it falls on depends on which knee was
    scanned. Coronal and axial views mirror under a horizontal flip. Sagittal stacks do
    not - each slice is unchanged by mirroring, what differs is the direction the stack
    traverses the joint - so the slice order is reversed instead.

    Where `Laterality` is absent the volume is left alone: a wrong flip is worse than no
    flip, and the presence mask lets the head learn how much to trust each slot.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


def available_ram_gb():
    """Physical RAM available to this process, in GB, or None if it cannot be read."""
    try:
        import psutil
        return psutil.virtual_memory().available / 1024 ** 3
    except Exception:
        pass
    try:                                   # Linux without psutil, which includes Kaggle
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    return int(line.split()[1]) / 1024 ** 2
    except Exception:
        pass
    return None


def plan_cache(n_study):
    """Choose how many slices per slot memory allows.

    The cache is n_study x n_slot x slices x img^2 bytes. It grows with the *square* of
    resolution and only linearly with slices, so coverage is the cheap axis and
    resolution the expensive one; when the budget binds it is the slice count that gives
    way. Deciding once, from the training corpus size, keeps train and test on the same
    group layout.

    The configured budget is a ceiling, not a promise. At full corpus size the cache is
    around 12 GB, which fits a 30 GB Kaggle instance and does not fit a 13 GB one - and
    the failure mode is a SIGKILL partway through the decode, which no `try` will catch
    and which costs the whole run. So the budget is also capped at a fraction of what
    the machine actually reports free, leaving room for the model, the reader queue and
    the test cache.
    """
    budget = CFG.cache_budget_gb
    free = available_ram_gb()
    if free is not None:
        safe = CFG.ram_fraction * free
        if safe < budget:
            log(f"cache budget trimmed {budget:.1f} -> {safe:.1f} GB "
                f"({free:.1f} GB free, keeping {1 - CFG.ram_fraction:.0%} headroom)")
            budget = safe
    per_slice = n_study * N_SLOT * CFG.img * CFG.img
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(CFG.n_group_max, afford // CFG.group))
    if groups < CFG.n_group_max:
        log(f"cache budget {budget:.1f} GB allows {groups} group(s) of "
            f"{CFG.group}, not {CFG.n_group_max}")
    return groups


def build_cache(slot_map, plane_map, lat_map, tag, n_group):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch, and a study is on the order of a
    hundred and fifty files. Reading them from the mount each epoch would make the epoch
    count a function of I/O rather than of learning.

    Reads are issued in bounded chunks: submitting every job at once lets the reader
    threads run arbitrarily far ahead and the completed buffers accumulate without limit.
    """
    cache_slices = CFG.group * n_group
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, cache_slices, CFG.img, CFG.img), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")

    # Both stat dicts are module-level, so without this the `test:` line reports
    # train+test cumulatively and its percentages describe neither pass.
    ORDER_STATS.update({k: 0 for k in ORDER_STATS})
    PIX_STATS.update({k: 0 for k in PIX_STATS})

    chunk = 512
    done = 0
    with ThreadPoolExecutor(max_workers=CFG.pix_threads) as pool:
        for c0 in range(0, len(jobs), chunk):
            block = jobs[c0:c0 + chunk]
            imgs = pool.map(lambda j: read_slot(j[3], cache_slices, CFG.img, j[2]), block)
            for (st, k, plane, _), img in zip(block, imgs):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < chunk:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > CFG.time_budget:
                log(f"  {tag}: time budget reached during decode")
                break
    total = sum(ORDER_STATS.values()) or 1
    log(f"{tag}: slice ordering " + ", ".join(
        f"{k} {v / total:.1%}" for k, v in ORDER_STATS.items() if v))
    if ORDER_STATS["filename"] / total > 0.05:
        warnings.warn(
            f"{ORDER_STATS['filename'] / total:.0%} of series fell back to filename "
            "order, which is SOP UID order and therefore random. Those slots carry no "
            "depth structure.", RuntimeWarning, stacklevel=2)

    n_crop = sum(PIX_STATS[k] for k in
                 ("crop_applied", "crop_padded", "crop_skipped_short_fov",
                  "crop_no_spacing")) or 1
    log(f"{tag}: physical scale " + ", ".join(
        f"{k.replace('crop_', '')} {PIX_STATS[k]} ({PIX_STATS[k] / n_crop:.1%})"
        for k in ("crop_applied", "crop_padded", "crop_skipped_short_fov",
                  "crop_no_spacing") if PIX_STATS[k]))
    log(f"{tag}: slices read {PIX_STATS['slices_read']}, "
        f"decode failures {PIX_STATS['decode_failed']}, "
        f"MONOCHROME1 inverted {PIX_STATS['mono1_inverted']}")
    skipped = PIX_STATS["crop_skipped_short_fov"] / n_crop
    if skipped > 0.05:
        warnings.warn(
            f"{skipped:.0%} of slots had a field of view at or below crop_mm="
            f"{CFG.crop_mm:.0f}mm, so no physical normalisation was applied to them and "
            "their mm/px came from the acquisition. Set RSNA_PAD_SHORT_FOV=1 to pad "
            "instead.", RuntimeWarning, stacklevel=2)
    gc.collect()
    return studies, cache, mask

In [ ]:
"""Private gold-only screen of an attributed RSNA-trained ConvNeXtV2 backbone."""

import copy
import gc
import hashlib
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, TensorDataset


TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
SEEDS = (20261007, 20261008, 20261009)
N_STUDIES, N_SLOT, N_DEPTH, FEATURE_DIM = 4407, 6, 10, 768
HEAD_EPOCHS = 45
CONTROL_GOLD_AUC = 0.860457
CHECKPOINT_NAME = "hunter_pool_v1_best.pt"
CHECKPOINT_SHA256 = "4680dff05e3556eebbF07e84ca824ab6f1fe8db442dc9cc3948e18d226f008c3".lower()
CACHE_SHA256 = "490e5ccc07143699d05a9eb3ef9583e08795f1c2bacc6d9cf2bd0a9268158526"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
START = time.time()


def log(message):
    print(f"[{time.time() - START:7.1f}s] {message}", flush=True)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sha256(path):
    value = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(16 * 1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()


def unique_input(name):
    matches = list(Path("/kaggle/input").rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f"expected exactly one {name}, found {matches}")
    return matches[0]


def competition_root():
    for root in (Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                 Path("/kaggle/input/rsna-knee-abnormality-detection")):
        if (root / "train.csv").is_file():
            return root
    raise FileNotFoundError("competition train.csv not mounted")


def macro_auc(truth, prediction):
    return float(np.mean([roc_auc_score(truth[:, i], prediction[:, i])
                          for i in range(truth.shape[1])]))


def report_groups(series):
    normalized = (series.fillna("").astype(str).str.lower()
                  .str.replace(r"\s+", " ", regex=True).str.strip())
    return np.asarray([hashlib.sha256(text.encode()).hexdigest()[:24]
                       for text in normalized])


def load_backbone(checkpoint_path):
    import timm

    state = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if not isinstance(state, dict) or "head.weight" not in state:
        raise RuntimeError("unexpected ConvNeXtV2 checkpoint container")
    if tuple(state["head.weight"].shape) != (12, 1536):
        raise RuntimeError(f"unexpected published pool head: {state['head.weight'].shape}")
    backbone_state = {key[2:]: value for key, value in state.items()
                      if key.startswith("0.")}
    model = timm.create_model("convnextv2_tiny", pretrained=False,
                              num_classes=0, global_pool="")
    missing, unexpected = model.load_state_dict(backbone_state, strict=False)
    allowed_missing = {"head.fc.weight", "head.fc.bias"}
    if set(missing) - allowed_missing or unexpected:
        raise RuntimeError(f"checkpoint/model mismatch: missing={missing}, unexpected={unexpected}")
    if int(model.num_features) != FEATURE_DIM:
        raise RuntimeError(f"feature dimension drift: {model.num_features}")
    model = model.to(DEVICE).eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    del state, backbone_state
    return model


@torch.no_grad()
def encode_dicom(model):
    """Reproduce the validated 288px cache contract shard-by-shard in memory."""
    output_path = Path("/kaggle/working/convnextv2_domain_features.npy")
    mask_path = Path("/kaggle/working/convnextv2_domain_masks.npy")
    study_path = Path("/kaggle/working/convnextv2_domain_studies.csv")
    features = np.lib.format.open_memmap(
        output_path, mode="w+", dtype=np.float16,
        shape=(N_STUDIES, N_SLOT, N_DEPTH, FEATURE_DIM))
    output_masks = np.zeros((N_STUDIES, N_SLOT), np.float32)
    mean = torch.tensor([.485, .456, .406], device=DEVICE)[None, :, None, None]
    std = torch.tensor([.229, .224, .225], device=DEVICE)[None, :, None, None]
    root = find_root()
    train_series = pd.read_csv(root / "train_series.csv")
    plane_map = dict(zip(train_series.SeriesInstanceUID, train_series.Anatomical_Plane))
    log("direct-DICOM header pass")
    headers = annotate(walk(root, "train_series"))
    headers["plane"] = headers.SeriesInstanceUID.map(plane_map)
    laterality = {study: resolve_laterality(group)
                  for study, group in headers.groupby("StudyInstanceUID")}
    slot_map = pick_slots(headers, plane_map)
    all_studies = sorted(slot_map)
    if len(all_studies) != N_STUDIES:
        raise RuntimeError(f"unexpected study count {len(all_studies)}")
    pd.DataFrame({"StudyInstanceUID": all_studies}).to_csv(study_path, index=False)

    shard_size = 32
    for lo in range(0, N_STUDIES, shard_size):
        hi = min(N_STUDIES, lo + shard_size)
        subset_ids = all_studies[lo:hi]
        subset = {uid: slot_map[uid] for uid in subset_ids}
        studies, cache, masks = build_cache(
            subset, plane_map, laterality, f"cnxv2-{lo:04d}", 4)
        if studies != subset_ids or cache.shape != (hi - lo, N_SLOT, 12, 288, 288):
            raise RuntimeError(f"direct cache identity/shape drift at {lo}:{hi}")
        output_masks[lo:hi] = masks
        tensor = torch.from_numpy(cache)
        windows = torch.stack((tensor[:, :, :-2], tensor[:, :, 1:-1],
                               tensor[:, :, 2:]), dim=3)
        token_mask = torch.from_numpy(masks).bool()[:, :, None].expand(-1, -1, N_DEPTH)
        images = windows[token_mask]
        encoded_parts = []
        for start in range(0, len(images), 32):
            image = images[start:start + 32].to(DEVICE).float().div_(255.0)
            image = (image - mean) / std
            with torch.autocast("cuda", dtype=torch.float16,
                                enabled=DEVICE.type == "cuda"):
                fmap = model(image.contiguous(memory_format=torch.channels_last))
                encoded = F.adaptive_avg_pool2d(fmap, 1).flatten(1)
            encoded_parts.append(encoded.float().cpu())
        encoded = torch.cat(encoded_parts).numpy().astype(np.float16)
        for location, value in zip(token_mask.nonzero(as_tuple=False).numpy(), encoded):
            batch_row, slot, depth = location
            features[lo + batch_row, slot, depth] = value
        features.flush()
        log(f"decoded and encoded {hi}/{N_STUDIES} studies")
        del cache, tensor, windows, images, encoded, encoded_parts
        gc.collect()
    features.flush()
    np.save(mask_path, output_masks)
    del features
    gc.collect()
    return output_path, mask_path, study_path


class StudyTransformerHead(nn.Module):
    """Public reader contract: slot/depth tokens, transformer, target attention."""

    def __init__(self, dim=384, depth=2, drop=.2):
        super().__init__()
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(FEATURE_DIM, dim),
                                  nn.LayerNorm(dim))
        self.slot = nn.Embedding(N_SLOT, dim)
        self.position = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=.1,
                                           batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, depth,
                                                  enable_nested_tensor=False)
        self.attention = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(),
                                       nn.Dropout(drop), nn.Linear(256, len(TARGETS)))
        self.classifier_weight = nn.Parameter(torch.randn(len(TARGETS), dim) * .02)
        self.classifier_bias = nn.Parameter(torch.zeros(len(TARGETS)))
        position = torch.linspace(1 / 11, 10 / 11, N_DEPTH)
        self.register_buffer("depth_position", position)

    def forward(self, feature, slot_mask):
        batch = feature.shape[0]
        slots = torch.arange(N_SLOT, device=feature.device)[None, :, None]
        positions = self.depth_position[None, None, :, None].expand(batch, N_SLOT, -1, -1)
        hidden = self.proj(feature) + self.slot(slots) + self.position(positions)
        hidden = hidden.reshape(batch, N_SLOT * N_DEPTH, -1)
        padding = ~slot_mask.bool()[:, :, None].expand(-1, -1, N_DEPTH).reshape(batch, -1)
        padding = padding & ~padding.all(1, keepdim=True)
        hidden = self.transformer(hidden, src_key_padding_mask=padding)
        score = self.attention(hidden).float().masked_fill(padding[:, :, None], -1e4)
        weights = score.softmax(1)
        pooled = torch.einsum("btc,btd->bcd", weights.to(hidden.dtype), hidden)
        return ((pooled * self.classifier_weight).sum(-1) + self.classifier_bias)


@torch.no_grad()
def predict(model, features, masks, indices):
    model.eval()
    dataset = TensorDataset(torch.from_numpy(np.asarray(features[indices], np.float32)),
                            torch.from_numpy(np.asarray(masks[indices], np.float32)))
    outputs = []
    for feature, mask in DataLoader(dataset, batch_size=128, shuffle=False):
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            outputs.append(torch.sigmoid(model(feature.to(DEVICE), mask.to(DEVICE)))
                           .float().cpu().numpy())
    return np.concatenate(outputs)


def train_head(features, masks, labels, fitting, validation, seed):
    seed_all(seed)
    model = StudyTransformerHead().to(DEVICE)
    ema = copy.deepcopy(model).eval()
    for parameter in ema.parameters():
        parameter.requires_grad_(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=.02)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HEAD_EPOCHS)
    dataset = TensorDataset(torch.from_numpy(np.asarray(features[fitting], np.float32)),
                            torch.from_numpy(np.asarray(masks[fitting], np.float32)),
                            torch.from_numpy(labels[fitting]))
    loader = DataLoader(dataset, batch_size=64, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    weak_truth = (labels[validation] >= .5).astype(np.int8)
    best_auc, best_state, history, step = -np.inf, None, [], 0
    for epoch in range(HEAD_EPOCHS):
        model.train()
        running = 0.0
        for feature, mask, target in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                logits = model(feature.to(DEVICE), mask.to(DEVICE))
                loss = F.binary_cross_entropy_with_logits(logits.float(), target.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            optimizer.step()
            step += 1
            decay = min(.998, (1 + step) / (10 + step))
            with torch.no_grad():
                for target_parameter, source_parameter in zip(ema.parameters(), model.parameters()):
                    target_parameter.lerp_(source_parameter, 1 - decay)
            running += float(loss.detach()) * len(feature)
        scheduler.step()
        weak_prediction = predict(ema, features, masks, validation)
        weak_auc = macro_auc(weak_truth, weak_prediction)
        history.append({"epoch": epoch + 1, "loss": running / len(dataset),
                        "weak_validation_auc": weak_auc})
        if weak_auc > best_auc:
            best_auc = weak_auc
            best_state = {key: value.detach().cpu().clone()
                          for key, value in ema.state_dict().items()}
        if (epoch + 1) % 10 == 0:
            log(f"seed {seed} epoch {epoch + 1}: weak AUC {weak_auc:.6f}")
    ema.load_state_dict(best_state)
    return ema, best_state, best_auc, history


def main():
    if DEVICE.type != "cuda":
        raise RuntimeError("GPU required")
    checkpoint = unique_input(CHECKPOINT_NAME)
    checkpoint_hash = sha256(checkpoint)
    if checkpoint_hash != CHECKPOINT_SHA256:
        raise RuntimeError(f"checkpoint hash drift: {checkpoint_hash}")
    train = pd.read_csv(competition_root() / "train.csv", dtype={"StudyInstanceUID": str})
    gold = train[TARGETS].notna().all(axis=1).to_numpy()
    if int(gold.sum()) != 58:
        raise RuntimeError("organizer gold count drift")
    labels_source = pd.read_csv(unique_input("llm_labels_v4_blend.csv"),
                                dtype={"StudyInstanceUID": str})
    aligned = train[["StudyInstanceUID"]].merge(labels_source, on="StudyInstanceUID",
                                                   how="left", validate="one_to_one")
    labels = aligned[TARGETS].to_numpy(np.float32)
    if not np.isfinite(labels).all():
        raise RuntimeError("Steven v4 labels incomplete")

    backbone = load_backbone(checkpoint)
    feature_path, mask_path, study_path = encode_dicom(backbone)
    del backbone
    torch.cuda.empty_cache()

    studies = pd.read_csv(study_path, dtype={"StudyInstanceUID": str})
    if len(studies) != N_STUDIES or not studies.StudyInstanceUID.is_unique:
        raise RuntimeError("direct-DICOM study manifest drift")
    order = pd.Series(np.arange(len(studies)), index=studies.StudyInstanceUID)
    reorder = order.loc[train.StudyInstanceUID].to_numpy()
    masks = np.load(mask_path)[reorder]
    features = np.load(feature_path, mmap_mode="r")[reorder]

    non_gold = np.flatnonzero(~gold)
    groups = report_groups(train.Report)
    train_local, validation_local = list(GroupKFold(5).split(
        non_gold, groups=groups[non_gold]))[0]
    fitting, validation = non_gold[train_local], non_gold[validation_local]
    if set(groups[fitting]).intersection(groups[validation]):
        raise RuntimeError("report-group leakage")
    gold_index = np.flatnonzero(gold)
    gold_truth = train.loc[gold, TARGETS].to_numpy(np.float32)

    rows, predictions, states = [], [], []
    for seed in SEEDS:
        model, state, weak_auc, history = train_head(
            features, masks, labels, fitting, validation, seed)
        gold_prediction = predict(model, features, masks, gold_index)
        gold_auc = macro_auc(gold_truth, gold_prediction)
        rows.append({"seed": seed, "best_weak_auc": weak_auc,
                     "gold_auc": gold_auc, "history": history})
        predictions.append(gold_prediction)
        states.append(state)
        log(f"seed {seed}: gold AUC {gold_auc:.6f}")

    ensemble_auc = macro_auc(gold_truth, np.mean(predictions, axis=0))
    heads_path = Path("/kaggle/working/convnextv2_domain_heads.pt")
    torch.save({"states": states, "seeds": SEEDS, "targets": TARGETS}, heads_path)
    report = {
        "contract": "private one-fold domain-backbone screen; no submission",
        "candidate": "frozen public RSNA ConvNeXtV2-Tiny backbone plus study Transformer head",
        "public_model": "yaolinghan/rsna-knee-pool-models/PyTorch/convnextv2-pool/1",
        "public_model_license": "Apache-2.0",
        "public_checkpoint": CHECKPOINT_NAME,
        "public_checkpoint_sha256": checkpoint_hash,
        "pixel_contract": "direct DICOM reproduction of validated 288px/140mm/12-slice cache",
        "reference_cache_h5_sha256": CACHE_SHA256,
        "feature_sha256": sha256(feature_path),
        "heads_sha256": sha256(heads_path),
        "label_source": "stevenleehans llm_labels_v4_blend.csv",
        "fit_studies": int(len(fitting)),
        "weak_validation_studies": int(len(validation)),
        "gold_fit_count": 0,
        "gold_evaluation_count": 58,
        "seeds": rows,
        "ensemble_gold_auc": ensemble_auc,
        "prior_private_control_auc": CONTROL_GOLD_AUC,
        "delta_vs_control": ensemble_auc - CONTROL_GOLD_AUC,
        "screen_passed": bool(ensemble_auc >= CONTROL_GOLD_AUC + .005 and
                              sum(row["gold_auc"] > CONTROL_GOLD_AUC for row in rows) >= 2),
        "submission_created": False,
    }
    Path("/kaggle/working/convnextv2_domain_validation_report.json").write_text(
        json.dumps(report, indent=2) + "\n")
    print(json.dumps(report, indent=2))


main()
